DH演習2026 · 教材 第53版 · 2026-r53 · 2026-11-25

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第53版
教材ID: 2026-r53
教材日付: 2026-11-25
活動: relation_network
環境: Colab / Jupyter
ソース: 3bb1cdc74699c6df82735f72bff2ce617a0b4964
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第11回：線を引く規則を変えると、誰が中心になるか

確認問題まで上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

確認問題の後の追加課題は授業外・任意です。予測とコードの変更は自分で行い、エラーが出たらAIに説明を求められます。

資料は芥川龍之介『二人小町』（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。関係表は教材の判断案です。

## 1. 本文を読む

ノートブックはTEIを取得し、カードA（第1場の発言14〜21）、B（第2場の発言16〜20）、C（第3場の発言1〜14）をルビの読みを除いて表示します。［ト書き］は舞台指示です。発言の途中にあるものも、話者が口にした言葉ではありません。

In [ ]:
#@title 操作：本文を取得して表示する
import xml.etree.ElementTree as ET

import networkx as nx
import requests

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
tei = ET.fromstring(requests.get(url).content)
for span in tei.iter(T + "span"):
    if span.get("type") in ("rt", "rp"):
        span.text = ""
def shown(e):
    # 発言の途中にある舞台指示にも印を付ける
    if e.tag == T + "stage":
        return "［ト書き］" + "".join(e.itertext())
    return (e.text or "") + "".join(shown(child) + (child.tail or "") for child in e)

cards = {"A": (1, 14, 21), "B": (2, 16, 20), "C": (3, 1, 14)}
for card, (scene, first, last) in cards.items():
    print("カード" + card)
    n = 0
    for e in tei.findall(f".//{T}div")[scene - 1]:
        if e.tag == T + "sp":
            n += 1
            speaker, *rest = e
            if first <= n <= last:
                print(f"  s{scene}-{n} {speaker.text}：" + "".join(shown(x) for x in rest))
        elif e.tag == T + "stage" and first <= n + 1 <= last:
            print("  ［ト書き］" + "".join(e.itertext()).strip())

## 2. 関係表を判断する

一つのカードでの一組ごとに、`co_mention`（共起：発言の話者と、その発言本文に書かれたフルネーム）と`interaction`（相互行為：対面の会話か直接の働きかけ）を、`yes`／`no`／`defer`で別々に判断してください。ト書きにある名前は話者が言った名前ではないので、共起に数えません。カードBのs2-16とs2-18で、使がフルネームを口にしているのはどちらかを確かめてください。R1・R2は記入例です。R3〜R7は本文と照合し、`basis`に箇所と理由を書いてください。

`unreviewed`は点検前の未判断です。点検しても資料が足りない場合は`defer`（保留）にし、理由を記してください。

In [ ]:
import pandas as pd

relations = pd.DataFrame([
    ("R1", "A", "ono", "tama", "yes", "no", "s1-15で小野が玉造のフルネームを言う。玉造は不在で、会話はない"),
    ("R2", "A", "ono", "messenger", "no", "yes", "発言にフルネームはない。s1-14〜20で身代わりを交渉する"),
    ("R3", "B", "messenger", "tama", "unreviewed", "unreviewed", ""),
    ("R4", "B", "messenger", "ono", "unreviewed", "unreviewed", ""),
    ("R5", "B", "ono", "tama", "unreviewed", "unreviewed", ""),
    ("R6", "C", "messenger", "guards", "unreviewed", "unreviewed", ""),
    ("R7", "C", "guards", "ono", "unreviewed", "unreviewed", ""),
], columns=["record_id", "card", "source", "target", "co_mention", "interaction", "basis"])
relations

## 3. 線を引く規則を選ぶ

ノードは四つの人物・集団、線は選んだ列が`yes`の行です。`use_interactions`を`True`にすると、共起の列から相互行為の列へ切り替わります。変える前に、消える線と増える線を予測してください。

In [ ]:
use_interactions = False

column = "interaction" if use_interactions else "co_mention"
g = nx.Graph()
g.add_nodes_from(["ono", "tama", "messenger", "guards"])
for row in relations.itertuples():
    if getattr(row, column) == "yes":
        g.add_edge(row.source, row.target)
print(column, list(g.edges))
nx.draw_networkx(g, pos=nx.circular_layout(g), node_color="lightgray", node_size=1600, margins=0.15)

表示された線のうち一つを選んでください。その線を加えた表の行は、`if`のどの条件を満たしていますか。

## 4. 次数中心性を手で求めて比べる

次数は線でつながる相手の数、次数中心性は次数÷（ノード数−1）です。一人の値を手で求めてから実行してください。

In [ ]:
centrality = nx.degree_centrality(g)
pd.DataFrame({"次数": dict(g.degree), "次数中心性": centrality})

この値は、今回の線の規則とカードの範囲で、線でつながる相手の多さを示します。文学的な重要性と読むには、本文の何を別に確かめる必要があるでしょうか。

## 確認問題

下の表は確認問題用の相互行為の記録です。ノードは小野の小町・玉造の小町・黄泉の使・神将の四つです。線を引く組は表の二組です。各組を紙に線で結び、`ono`の次数中心性を計算して`answer`に入れてください。

In [ ]:
check_edges = [("ono", "messenger"), ("guards", "ono")]
pd.DataFrame(check_edges, columns=["source", "target"])

In [ ]:
answer = None

In [ ]:
#@title 答えを確かめる
hints = [
    (2 / 3, ""),
    (1 / 3, "表の二組のうち、onoとつながる相手を一人ずつ数えてください"),
    (2, "それは次数です。次数中心性は次数÷（全ノード数−1）です"),
    (0.5, "線のない人物も含めて、ノードは4つです"),
    (1, "全員がonoとつながるか、表の二組を確かめてください"),
]
if answer is None:
    print("結果：不合格（`answer`に数を入れてください）")
else:
    hint = next((h for v, h in hints if abs(answer - v) < 0.006), "`ono`と線でつながる相手を数え、全ノード数−1で割ってください")
    print(f"結果：不合格（{hint}）" if hint else "結果：合格")

## 授業外・任意：自分の判断からできた線と比べる

3節で相互行為の図を作り、どの相手が確認問題と共通するか予測してください。下のセルで自分の図の隣接ノードを表示し、線の違いを一つ、関係表の行と本文の箇所に結び付けて説明してください。同じ相手が並んだら、そのうち一人を選んで裏付けを示してください。

別のノードでも予測し、下の`node`を変えて調べてください。

In [ ]:
node = "ono"
print("選んだ列：", column)
print(f"{node}につながる相手：", sorted(g.neighbors(node)))

## 授業外・任意：向きのある関係と対称な関係を分ける

共起の行は「話者（`source`）が相手（`target`）のフルネームを言う」という向きのある関係です。相互行為は、一方の向きで成り立てば逆の向きでも成り立つ対称な関係です。関係が対称かどうかも、オントロジーが定める性質の一つです。下のセルは`relation`で選んだ列から有向の図を作り、`symmetric`で対称とした関係にだけ逆向きの線を加えます。

`relation`を共起のままにして実行し、R1からできた線の始点と終点を確かめてください。`symmetric`の`"co_mention"`を`True`に変えると、どの線が加わるでしょうか。予測してから変更し、加わった線が表す文がカードAの本文で成り立つかを照合してください。

In [ ]:
relation = "co_mention"
symmetric = {"co_mention": False, "interaction": True}

directed = nx.DiGraph()
directed.add_nodes_from(g.nodes)
for row in relations.itertuples():
    if getattr(row, relation) == "yes":
        directed.add_edge(row.source, row.target)
        if symmetric[relation]:
            directed.add_edge(row.target, row.source)
print("有向の線：", sorted(directed.edges))
pd.DataFrame({"始点になる線": dict(directed.out_degree), "終点になる線": dict(directed.in_degree)})